# Urban Informatics Final Project: Data Cleaning

Spring 2026

**Contributors:** Anwar Baroudi, Phoebe Chiu, Noelani Fixler, Michael Huang

### **1. Setting Up the Document**

In [5]:
# Import the libraries and modules we need
import pandas as pd
import numpy as np
import math
import os
import geopandas as gpd
from tabulate import tabulate
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import shutil
import requests
from google.colab import drive
drive.mount('/content/drive')
%run "/content/drive/MyDrive/Urban Informatics Final Project/Colab Notebooks/functions.ipynb"
##^^create shortcut to your drive for the urban informatics final project folder to make this work

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **2. Organizing Equity Priority Communities Data**

In [9]:
# Get the Census tracts for all Equity Priority Communities
# Source: https://opendata.mtc.ca.gov/datasets/equity-priority-communities-plan-bay-area-2050/explore?location=37.727536%2C-122.235343%2C12

# Retrieve data from Github
epc_url = "https://github.com/MiAnPh/MAPN-SRTS/raw/main/data/raw/equity_priority_communities_2020_acs2018.zip"

# Pull Github data from URL
mtc_epcs = gpd.read_file(epc_url)

# Clean column names
mtc_epcs.columns = (
    mtc_epcs.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

### **3. Organizing Spatial Data for Schools**

#### *a) Explore California Open Data Portal School Data*

In [10]:
# Organize the school datasets
# Sources:
# https://data.ca.gov/dataset/california-private-schools-2023-24
# https://data.ca.gov/dataset/california-public-schools-2023-24

# Get the uploaded data from Github
ca_pub_schools_url = 'https://github.com/MiAnPh/MAPN-SRTS/raw/main/data/raw/California%20Public%20Schools%202023-2024.geojson'
ca_private_schools_url = 'https://github.com/MiAnPh/MAPN-SRTS/raw/main/data/raw/California%20Private%20Schools%202023-2024.geojson'

# Read GeoJSON files with Github links
ca_public_schools = gpd.read_file(ca_pub_schools_url)
ca_private_schools = gpd.read_file(ca_private_schools_url)

# Immediately filtering just for county
ac_public_schools = ca_public_schools[ca_public_schools['CountyName'] == "Alameda"]
ac_private_schools = ca_private_schools[ca_private_schools['County'] == "Alameda"]

#### *b) Create truncated dataframes to contacenate*

In [11]:
# Create a list of variables for final project
final_public_school_vars = ['CountyName', 'SchoolName', 'City', 'GradeLow', 'GradeHigh', 'Latitude', 'Longitude',
                            'EnrollTotal', 'geometry']

final_private_school_vars = ['County', 'SchoolName', 'City', 'LowGrade', 'HighGrade', 'Lat', 'Lon', 'geometry',
                             'TotalEnrol']

In [12]:
# Create the truncated dataframes
ac_pub_schools = ac_public_schools[final_public_school_vars]
ac_priv_schools = ac_private_schools[final_private_school_vars]

In [13]:
# Rename the column with county information in `ca_pub_schools`
ac_pub_schools = ac_pub_schools.rename(columns={
    'CountyName': 'County'
})

In [14]:
# Rename the `ca_priv_schools` columns to match the public school column names
ac_priv_schools = ac_priv_schools.rename(columns={
    'LowGrade': 'GradeLow',
    'HighGrade': 'GradeHigh',
    'Lat': 'Latitude',
    'Lon': 'Longitude',
    'TotalEnrol': 'EnrollTotal'
})

In [15]:
# Create a new column for each dataframe indicating school type
ac_pub_schools['Type'] = 'Public'
ac_priv_schools['Type'] = 'Private'

In [16]:
# Concacenate the two dataframes
ac_schools = pd.concat([ac_pub_schools, ac_priv_schools])

In [17]:
# Sort the dataframe alphabetically
ac_schools = ac_schools.sort_values('SchoolName', ascending=True)

In [18]:
# Create a dataframe of schools in the cities/unincorporated areas representing our project
ac_schools['City'].unique()

array(['Oakland', 'Livermore', 'Alameda', 'Hayward', 'San Leandro',
       'Albany', 'Pleasanton', 'Fremont', 'Union City', 'Berkeley',
       'Emeryville', 'San Lorenzo', 'Newark', 'Piedmont', 'Castro Valley',
       'Dublin', 'Byron', 'Sunol'], dtype=object)

### **4. Filter for Schools Within EPC Study Areas**

#### *a) Filter School Data for Alameda County Only*

Only keeping Alameda County schools because Tile2Net is only supported in Alameda County.

In [19]:
# Sort the dataframe alphabetically
ac_schools = ac_schools.sort_values('SchoolName', ascending=True)

#### *b) Conduct a Within Spatial Join*

In [20]:
# Adjust the CRS of schools to the `mtc_epcs` CRS 4326
mtc_ac_schools = ac_schools.to_crs(mtc_epcs.crs)

In [21]:
# Conduct a spatial join
proj_schools_gdf = gpd.sjoin(mtc_ac_schools, mtc_epcs, how='inner', predicate='within')

### **5. Finalize Dataframes for Upload**

#### *a) Filter Public School Data for Alameda County*

Only keeping Alameda County schools because Tile2Net is only supported in Alameda County.

In [22]:
# Create lists of variables to keep for each of the two dataframes
# Save this list for the future for race/ethnicity or enrollment levels analysis
public_school_vars = ['Year', 'FedID', 'Region',
       'CountyName', 'DistrictName', 'SchoolName', 'SchoolType', 'Status',
       'OpenDate', 'ClosedDate', 'SchoolLevel', 'GradeLow', 'GradeHigh',
       'Charter', 'Virtual', 'Magnet','TitleIStatus', 'Street', 'City', 'Zip',
       'State', 'Latitude', 'Longitude', 'EnrollTotal', 'AAcount', 'AApct', 'AIcount', 'AIpct',
       'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount',
       'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct',
       'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct',
       'MIGcount', 'MIGpct', 'SEDCount', 'SEDpct', 'SWDcount', 'SWDpct',
       'FRPMcount', 'FRPMpct', 'geometry']

In [23]:
ac_schools_public = ac_public_schools[public_school_vars]

#### *b) Conduct a Within Spatial Join*

In [24]:
# Adjust the CRS of schools to the `mtc_epcs` CRS
mtc_ac_schools_public = ac_schools_public.to_crs(mtc_epcs.crs)
mtc_ac_schools_private = ac_priv_schools.to_crs(mtc_epcs.crs)
mtc_ac_schools_all = ac_schools.to_crs(mtc_epcs.crs)

In [25]:
# Conduct spatial joins
proj_schools_public_gdf = gpd.sjoin(mtc_ac_schools_public, mtc_epcs, how='inner', predicate='within')
proj_schools_private_gdf = gpd.sjoin(mtc_ac_schools_private, mtc_epcs, how='inner', predicate='within')
proj_schools_all_gdf = gpd.sjoin(mtc_ac_schools_all, mtc_epcs, how='inner', predicate='within')

In [26]:
# changing 'inner' to 'left' to keep all Alameda county schools
mtc_ac_schools_all = ac_schools.to_crs(mtc_epcs.crs)
ac_schools_with_epc_flag = gpd.sjoin(mtc_ac_schools_all, mtc_epcs, how='left', predicate='within')
# + creating a boolean flag based on whether a match was found (index_right = NaN for non-EPC schools)
ac_schools_with_epc_flag['within_epc'] = ac_schools_with_epc_flag['index_right'].notnull()

#### *c) Upload Dataframes to Github*

In [ ]:
#using github token to reeupload directly to git in [data/processed]
upload_gdf_to_github(proj_schools_public_gdf, "proj_schools_public.geojson")

upload_gdf_to_github(proj_schools_private_gdf, "proj_schools_private.geojson")

upload_gdf_to_github(proj_schools_all_gdf, "proj_schools_all.geojson")

upload_gdf_to_github(ac_schools_with_epc_flag, "ac_schools_full_with_flag.geojson")

 saved proj_schools_public.geojson locally
Cloning into 'MAPN'...
remote: Enumerating objects: 343, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 343 (delta 90), reused 20 (delta 11), pack-reused 197 (from 2)
Receiving objects: 100% (343/343), 15.46 MiB | 13.90 MiB/s, done.
Resolving deltas: 100% (158/158), done.
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
success: proj_schools_public.geojson now in MAPN/data/processed
 saved proj_schools_private.geojson locally
Cloning into 'MAPN'...
remote: Enumerating objects: 343, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 343 (delta 90), reused 20 (delta 11), pack-reused 197 (from 2)
Receiving objects: 100% (343/343), 15.46 MiB | 18.58 MiB/s, done.
Resolving deltas: 100% (158/158), done.
On branch main
Your branch is up to date with 'origin/main'